# CSE425: GNN-BERT Music Context Understanding — Demo Notebook
This interactive notebook demonstrates the end-to-end inference pipeline for a single music track using the trained **Cross-Attention Fusion** model.

### Pipeline Overview:
$$\text{Raw Audio} \xrightarrow{\text{Extract}} \text{Log-Mel / Chroma} \xrightarrow{\text{Graph Builder}} \mathcal{G}=(V,E) \xrightarrow{\text{GNN}} g$$
$$\text{Text / Tags} \xrightarrow{\text{Tokenize}} \text{BERT} \xrightarrow{} H_{\text{text}}$$
$$\text{Cross Attention}(Q=g, K=H_{\text{text}}, V=H_{\text{text}}) \to z \to \text{Context Predictions}$$

In [ ]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch_geometric.data import Batch

# Ensure project root is available
project_root = os.path.abspath(".." if os.path.exists("../src") else ".")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.utils import get_device, load_config, resolve_paths, load_checkpoint, get_genre_labels
from src.audio_features import extract_audio_features
from src.graph_builder import build_graph_from_features
from src.fusion_model import CrossAttentionFusion
from transformers import AutoTokenizer

sns.set_theme(style="whitegrid")
device = get_device()
print(f"Active device: {device}")

In [ ]:
# 1. Load Configuration
config = load_config("config.yaml")
resolve_paths(config, project_root)
genre_labels = get_genre_labels()
print(f"Target Label Vocabulary ({len(genre_labels)} classes): {genre_labels}")

In [ ]:
# 2. Audio Preprocessing & Graph Construction Function
def process_audio_to_graph(audio_path, config):
    ds = config["dataset"]
    segments = extract_audio_features(
        audio_path,
        sr=ds["sample_rate"],
        segment_length_s=ds["segment_length_s"],
        n_mels=ds["n_mels"],
        n_fft=ds["n_fft"],
        hop_length=ds["hop_length"],
        n_chroma=ds["n_chroma"],
        max_segments=ds["max_segments_per_track"],
    )
    graph = build_graph_from_features(
        segments,
        similarity_threshold=config["graph"]["similarity_threshold"],
        temporal_weight=config["graph"]["temporal_edge_weight"],
        include_chroma=config["graph"]["include_chroma_features"],
    )
    return segments, graph

In [ ]:
# 3. Run Inference on a Sample Track
# Example sample metadata
sample_text = "Genre: Hip-Hop. Title: Food. Tags: awol, underground, beat, rap. Artist: AWOL."
sample_audio_path = os.path.join(config["dataset"]["raw_audio_dir"], "000", "000002.mp3")

print(f"Input Text Context: '{sample_text}'")
print(f"Audio file exists: {os.path.exists(sample_audio_path)}")

if os.path.exists(sample_audio_path):
    segments, graph = process_audio_to_graph(sample_audio_path, config)
else:
    print("Audio file not found locally — creating synthetic segment graph for demonstration...")
    import soundfile as sf
    import tempfile
    t = np.linspace(0, 5, int(22050 * 5), endpoint=False)
    synth_wav = (0.5 * np.sin(2 * np.pi * 440 * t)).astype(np.float32)
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        sf.write(tmp.name, synth_wav, 22050)
        segments, graph = process_audio_to_graph(tmp.name, config)
        os.remove(tmp.name)

print(f"Graph constructed: {graph.num_nodes} segment nodes, {graph.edge_index.size(1)} edges")

In [ ]:
# 4. Tokenize Text and Build Model
tokenizer = AutoTokenizer.from_pretrained(config["bert"]["model_name"])
text_enc = tokenizer(
    sample_text,
    padding="max_length",
    truncation=True,
    max_length=config["bert"]["max_length"],
    return_tensors="pt",
)

pyg_batch = Batch.from_data_list([graph]).to(device)
input_ids = text_enc["input_ids"].to(device)
attention_mask = text_enc["attention_mask"].to(device)

in_channels = graph.x.size(-1)
model = CrossAttentionFusion.from_config(config, gnn_in_channels=in_channels).to(device)

# Attempt to load trained checkpoint if available
# The trainer copies the best checkpoint of each task into the canonical
# checkpoints/ directory (as well as results/<task>_<timestamp>/), so look
# there first and only fall back to scanning results/ for older runs.
import glob
ckpt_path = os.path.join(config["paths"]["checkpoints"], "fusion_best.pt")
if not os.path.exists(ckpt_path):
    candidates = sorted(glob.glob(os.path.join(config["paths"]["results"], "**", "fusion_best.pt"), recursive=True))
    ckpt_path = candidates[-1] if candidates else ckpt_path

if os.path.exists(ckpt_path):
    print(f"Loading trained checkpoint from {ckpt_path}")
    load_checkpoint(ckpt_path, model, device=device)
else:
    print("Note: No trained checkpoint found. Running forward pass with initialized weights.")

model.eval()
with torch.no_grad():
    logits, attn_weights = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        x=pyg_batch.x,
        edge_index=pyg_batch.edge_index,
        edge_attr=pyg_batch.edge_attr,
        batch=pyg_batch.batch,
        return_attention=True,
    )
    probs = torch.sigmoid(logits).squeeze(0).cpu().numpy()

In [ ]:
# 5. Display Predicted Context Labels & Probabilities
plt.figure(figsize=(9, 4.5))
colors = ["#2b5c8f" if p > 0.5 else "#a0aec0" for p in probs]
bars = plt.barh(genre_labels, probs, color=colors)
plt.axvline(0.5, color="red", linestyle="--", label="Decision Threshold (0.5)")
plt.xlabel("Predicted Probability")
plt.title("GNN-BERT Cross-Attention Context Predictions")
plt.xlim(0, 1)
for bar, p in zip(bars, probs):
    plt.text(p + 0.02, bar.get_y() + bar.get_height()/2, f"{p:.3f}", va="center", fontsize=9)
plt.legend()
plt.tight_layout()
plt.show()